# 15_E7 - Fast candidates patch

Este notebook reemplaza la celda pesada de `Strict candidates` del notebook 15.

Uso recomendado:
1. Correr el notebook `15_E7_alkafri_axial_license_and_curated_subset_fixed_v2` hasta que genere `image_case_index_df` y `gt_case_index_df`, o hasta que existan los CSV en `results/E7_alkafri_axial_curated_subset`.
2. Detener la celda lenta.
3. Correr este notebook.

La mejora principal es que NO lee shapes DICOM/PNG dentro del loop de 5000 candidatos. Primero genera candidatos por metadata y recién lee imágenes/máscaras para una muestra visual controlada.


In [1]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])

ensure_package('pydicom')
ensure_package('skimage', 'scikit-image')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.transform import resize
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('Drive no montado automáticamente:', repr(exc))

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 140)


Mounted at /content/drive


In [2]:
def find_pfi_root():
    expected = Path('/content/drive/MyDrive/PFI_MVP')
    if (expected / 'results/E7_alkafri_axial_curated_subset/E7_alkafri_axial_image_case_index.csv').exists():
        return expected
    for base in [Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')]:
        if not base.exists():
            continue
        for marker_name in ['E7_alkafri_axial_image_case_index.csv', 'E6_alkafri_image_specific_tokens.csv']:
            matches = list(base.rglob(marker_name))
            if matches:
                parts = matches[0].parts
                if 'results' in parts:
                    return Path(*parts[:parts.index('results')])
    return expected

PFI_ROOT = find_pfi_root()
CURATION_ROOT = PFI_ROOT / 'results' / 'E7_alkafri_axial_curated_subset'
FIGURES_ROOT = PFI_ROOT / 'figures'
DOCS_ROOT = PFI_ROOT / 'docs'
CURATION_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
DOCS_ROOT.mkdir(parents=True, exist_ok=True)

print('PFI_ROOT:', PFI_ROOT)
print('CURATION_ROOT:', CURATION_ROOT)

image_index_path = CURATION_ROOT / 'E7_alkafri_axial_image_case_index.csv'
gt_index_path = CURATION_ROOT / 'E7_alkafri_gt_case_index.csv'

if not image_index_path.exists() or not gt_index_path.exists():
    raise FileNotFoundError('Faltan los índices E7. Primero correr el notebook 15 hasta la celda que genera image_case_index_df y gt_case_index_df.')

image_case_index_df = pd.read_csv(image_index_path, low_memory=False)
gt_case_index_df = pd.read_csv(gt_index_path, low_memory=False)

print('image_case_index_df:', image_case_index_df.shape)
print('gt_case_index_df:', gt_case_index_df.shape)
display(image_case_index_df.head())
display(gt_case_index_df.head())


PFI_ROOT: /content/drive/MyDrive/PFI_MVP
CURATION_ROOT: /content/drive/MyDrive/PFI_MVP/results/E7_alkafri_axial_curated_subset
image_case_index_df: (8150, 8)
gt_case_index_df: (37080, 8)


,image_file_path,image_relative_path,case_id,modality,series_id,series_description,instance_number,is_posdisp_or_localizer
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_001.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,1,False
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_002.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,2,False
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_003.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,3,False
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_004.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,4,False
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_...,_nested/main_dataset__MRI_Data/01_MRI_Data/0037/L-SPINE_LSS_20150919_125616_225000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0037_005.ima,37,T2,1.3.12.2.1107.5.2.40.50233.2015091913064063223406200.0.0.0,t2_tse_tra_384,5,False


,gt_file_path,gt_relative_path,case_id,modality,disc_or_slice_id,labeller,gt_type,extension
0,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_...,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D3.png,1,T1,3,1.0,manual,.png
1,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_...,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D3.xcf,1,T1,3,1.0,manual,.xcf
2,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_...,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D4.png,1,T1,4,1.0,manual,.png
3,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_...,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D4.xcf,1,T1,4,1.0,manual,.xcf
4,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_...,_nested/ground_truth__Manual_Label_Data/03_Manual_Label_Data/Labeller 01/T1_0001_D5.png,1,T1,5,1.0,manual,.png


In [3]:
MAX_IMAGES_PER_CASE_MODALITY_FAST = 6
MAX_GT_PER_CASE_MODALITY_FAST = 8
MAX_FAST_CANDIDATES = 1200
MAX_VIS_CANDIDATES_FAST = 80
RANDOM_SEED = 42

image_cases = set(image_case_index_df['case_id'].dropna().astype(str))
gt_cases = set(gt_case_index_df['case_id'].dropna().astype(str))
common_cases = sorted(image_cases & gt_cases)
print('Casos imagen:', len(image_cases))
print('Casos GT:', len(gt_cases))
print('Casos en común:', len(common_cases))
print('Primeros casos comunes:', common_cases[:30])

image_case_modality_df = image_case_index_df.dropna(subset=['case_id', 'modality']).groupby(['case_id', 'modality']).size().reset_index(name='n_images')
gt_case_modality_df = gt_case_index_df.dropna(subset=['case_id', 'modality']).groupby(['case_id', 'modality']).size().reset_index(name='n_gt')
case_modality_overlap_df = image_case_modality_df.merge(gt_case_modality_df, on=['case_id', 'modality'], how='inner')
case_modality_overlap_df.to_csv(CURATION_ROOT / 'E7_alkafri_case_modality_overlap_diagnosis_FAST.csv', index=False)
print('Cruces caso+modalidad:', len(case_modality_overlap_df))
display(case_modality_overlap_df.head(30))

image_filtered_df = image_case_index_df[
    image_case_index_df['case_id'].notna()
    & image_case_index_df['modality'].notna()
    & ~image_case_index_df['is_posdisp_or_localizer'].fillna(False)
].copy()

gt_filtered_df = gt_case_index_df[
    gt_case_index_df['case_id'].notna()
    & gt_case_index_df['modality'].notna()
    & gt_case_index_df['extension'].astype(str).str.lower().eq('.png')
    & gt_case_index_df['gt_type'].isin(['final', 'intermediary'])
].copy()

print('image_filtered:', len(image_filtered_df))
print('gt_filtered:', len(gt_filtered_df))

# Priorizamos T1 porque el ground truth axial documentado aparece mayormente como T1.
image_filtered_df['modality_priority'] = image_filtered_df['modality'].map({'T1': 0, 'T2': 1}).fillna(9)
gt_filtered_df['gt_type_priority'] = gt_filtered_df['gt_type'].map({'final': 0, 'intermediary': 1}).fillna(9)
gt_filtered_df['modality_priority'] = gt_filtered_df['modality'].map({'T1': 0, 'T2': 1}).fillna(9)

image_reduced_df = (
    image_filtered_df
    .sort_values(['case_id', 'modality_priority', 'modality', 'instance_number'])
    .groupby(['case_id', 'modality'], group_keys=False)
    .head(MAX_IMAGES_PER_CASE_MODALITY_FAST)
    .copy()
)
gt_reduced_df = (
    gt_filtered_df
    .sort_values(['case_id', 'modality_priority', 'modality', 'gt_type_priority', 'disc_or_slice_id'])
    .groupby(['case_id', 'modality'], group_keys=False)
    .head(MAX_GT_PER_CASE_MODALITY_FAST)
    .copy()
)

candidate_base_df = image_reduced_df.merge(gt_reduced_df, on=['case_id', 'modality'], how='inner', suffixes=('_img', '_gt'))
print('candidate_base inicial FAST:', len(candidate_base_df))

if len(candidate_base_df) > MAX_FAST_CANDIDATES:
    candidate_base_df = candidate_base_df.sample(n=MAX_FAST_CANDIDATES, random_state=RANDOM_SEED).copy()
    print('candidate_base recortado FAST:', len(candidate_base_df))

strict_candidate_pairs_fast_df = candidate_base_df[[
    'image_file_path', 'gt_file_path', 'image_relative_path', 'gt_relative_path',
    'case_id', 'modality', 'gt_type', 'disc_or_slice_id', 'instance_number'
]].copy()
strict_candidate_pairs_fast_df['image_shape'] = ''
strict_candidate_pairs_fast_df['mask_shape'] = ''
strict_candidate_pairs_fast_df['evidence_case_match'] = True
strict_candidate_pairs_fast_df['evidence_modality_match'] = True
strict_candidate_pairs_fast_df['evidence_slice_match'] = strict_candidate_pairs_fast_df['disc_or_slice_id'].notna()
strict_candidate_pairs_fast_df['evidence_shape_match'] = ''
strict_candidate_pairs_fast_df['confidence_previsual'] = 'pendiente_visual'
strict_candidate_pairs_fast_df['needs_manual_review'] = True
strict_candidate_pairs_fast_df['reason'] = 'FAST: no se leen shapes en esta etapa; se valida en visual/sanity para evitar tiempos excesivos en Drive.'

fast_candidates_path = CURATION_ROOT / 'E7_alkafri_axial_strict_candidate_pairs_FAST.csv'
strict_candidate_pairs_fast_df.to_csv(fast_candidates_path, index=False)
print('Strict candidates FAST:', len(strict_candidate_pairs_fast_df))
print('CSV:', fast_candidates_path)
display(strict_candidate_pairs_fast_df.head())


Casos imagen: 185
Casos GT: 515
Casos en común: 185
Primeros casos comunes: ['1', '10', '100', '101', '103', '104', '105', '106', '107', '108', '109', '11', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128']
Cruces caso+modalidad: 369


,case_id,modality,n_images,n_gt
0,1,T1,12,48
1,1,T2,15,3
2,2,T1,17,48
3,2,T2,15,3
4,3,T1,15,48
5,3,T2,18,3
6,4,T1,15,48
7,4,T2,18,3
8,5,T1,15,48
9,5,T2,18,3


image_filtered: 7469
gt_filtered: 3090
candidate_base inicial FAST: 6624
candidate_base recortado FAST: 1200
Strict candidates FAST: 1200
CSV: /content/drive/MyDrive/PFI_MVP/results/E7_alkafri_axial_curated_subset/E7_alkafri_axial_strict_candidate_pairs_FAST.csv


,image_file_path,gt_file_path,image_relative_path,gt_relative_path,case_id,modality,gt_type,disc_or_slice_id,instance_number,image_shape,mask_shape,evidence_case_match,evidence_modality_match,evidence_slice_match,evidence_shape_match,confidence_previsual,needs_manual_review,reason
96,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0003/L-SPINE_LSS_20160411_134646_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,_nested/main_dataset__MRI_Data/01_MRI_Data/0003/L-SPINE_LSS_20160411_134646_314000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0003_003.ima,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0003_D3.png,3,T2,intermediary,3,3,,,True,True,True,,pendiente_visual,True,FAST: no se leen shapes en esta etapa; se valida en visual/sanity para evitar tiempos excesivos en Drive.
994,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0030/L-SPINE_LSS_20160319_123721_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,_nested/main_dataset__MRI_Data/01_MRI_Data/0030/L-SPINE_LSS_20160319_123721_224000/T2_TSE_TRA_384_0004/T2_TSE_TRA__0030_002.ima,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0030_D4.png,30,T2,intermediary,4,2,,,True,True,True,,pendiente_visual,True,FAST: no se leen shapes en esta etapa; se valida en visual/sanity para evitar tiempos excesivos en Drive.
1400,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0041/L-SPINE_LSS_20160112_154007_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,_nested/main_dataset__MRI_Data/01_MRI_Data/0041/L-SPINE_LSS_20160112_154007_572000/T2_TSE_TRA_384_0005/T2_TSE_TRA__0041_005.ima,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T2_Output/T2_0041_D5.png,41,T2,intermediary,5,5,,,True,True,True,,pendiente_visual,True,FAST: no se leen shapes en esta etapa; se valida en visual/sanity para evitar tiempos excesivos en Drive.
865,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0026/L-SPINE_LSS_20151121_130729_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,_nested/main_dataset__MRI_Data/01_MRI_Data/0026/L-SPINE_LSS_20151121_130729_925000/T1_TSE_TRA_0005/T1_TSE_TRA__0026_001.ima,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0026_D4.png,26,T1,intermediary,4,1,,,True,True,True,,pendiente_visual,True,FAST: no se leen shapes en esta etapa; se valida en visual/sanity para evitar tiempos excesivos en Drive.
6099,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0186/L-SPINE_CLINICAL_LIBRARIES_2...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,_nested/main_dataset__MRI_Data/01_MRI_Data/0186/L-SPINE_CLINICAL_LIBRARIES_20160512_102123_478000/T1_TSE_TRA_0005/T1_TSE_TRA__0186_006.ima,_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T1_Output/T1_0186_D3.png,186,T1,intermediary,3,6,,,True,True,True,,pendiente_visual,True,FAST: no se leen shapes en esta etapa; se valida en visual/sanity para evitar tiempos excesivos en Drive.


In [4]:
def read_dicom_pixels(path):
    ds = pydicom.dcmread(str(path), force=True)
    return ds.pixel_array.astype(np.float32), ds

def normalize_image(arr):
    arr = np.asarray(arr, dtype=np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros_like(arr, dtype=np.float32)
    arr = np.clip(arr, p1, p99)
    return (arr - p1) / (p99 - p1 + 1e-8)

def read_mask_array(path):
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3:
        return np.any(arr[..., :3] > 0, axis=-1).astype(np.uint8)
    return (arr > 0).astype(np.uint8)

def resize_mask_for_display(mask, target_shape):
    if tuple(mask.shape) == tuple(target_shape):
        return mask
    resized = resize(mask.astype(np.float32), target_shape, order=0, preserve_range=True, anti_aliasing=False)
    return (resized > 0.5).astype(np.uint8)

def mask_sanity(mask):
    fg = mask > 0
    ratio = float(fg.mean())
    _, ncomp = ndimage.label(fg)
    return {
        'mask_empty': bool(ratio == 0.0),
        'mask_full': bool(ratio > 0.95),
        'foreground_ratio': ratio,
        'component_count': int(ncomp),
    }

vis_df = strict_candidate_pairs_fast_df.head(MAX_VIS_CANDIDATES_FAST).copy()
figure_rows = []
sanity_rows = []
print('Candidatos para visualización FAST:', len(vis_df))

for i, (_, row) in enumerate(tqdm(vis_df.iterrows(), total=len(vis_df), desc='FAST visual sanity'), start=1):
    candidate_id = f'fast_candidate_{i:03d}'
    image_path = row['image_file_path']
    gt_path = row['gt_file_path']
    sanity = {
        'candidate_id': candidate_id,
        'image_file_path': image_path,
        'gt_file_path': gt_path,
        'case_id': row.get('case_id'),
        'modality': row.get('modality'),
        'gt_type': row.get('gt_type'),
        'disc_or_slice_id': row.get('disc_or_slice_id'),
        'instance_number': row.get('instance_number'),
        'overlay_generated': False,
        'auto_sanity_status': 'error',
    }
    try:
        img, _ = read_dicom_pixels(image_path)
        img_norm = normalize_image(img)
        mask = read_mask_array(gt_path)
        mask_for_display = resize_mask_for_display(mask, img.shape)
        ms = mask_sanity(mask_for_display)
        sanity.update(ms)
        sanity['image_shape_actual'] = str(tuple(img.shape))
        sanity['mask_shape_actual'] = str(tuple(mask.shape))
        sanity['resize_needed'] = bool(tuple(mask.shape) != tuple(img.shape))
        sanity['overlay_generated'] = True
        auto_ok = (not ms['mask_empty'] and not ms['mask_full'] and 0.0005 <= ms['foreground_ratio'] <= 0.70)
        sanity['auto_sanity_status'] = 'ok' if auto_ok else 'review'

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_norm, cmap='gray'); axes[0].set_title('Axial IMA'); axes[0].axis('off')
        axes[1].imshow(mask_for_display, cmap='gray'); axes[1].set_title('Mask/GT'); axes[1].axis('off')
        axes[2].imshow(img_norm, cmap='gray')
        axes[2].imshow(np.ma.masked_where(mask_for_display <= 0, mask_for_display), alpha=0.45, cmap='autumn')
        axes[2].set_title('Overlay'); axes[2].axis('off')
        fig.suptitle(f"{candidate_id} | case={row.get('case_id')} | {row.get('modality')} | gt={row.get('gt_type')} | D={row.get('disc_or_slice_id')} | inst={row.get('instance_number')}", fontsize=10)
        fig.tight_layout()
        fig_path = FIGURES_ROOT / f'E7_alkafri_fast_curated_candidate_{i:03d}.png'
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        figure_rows.append({
            'candidate_id': candidate_id,
            'figure_path': str(fig_path),
            'image_file_path': image_path,
            'gt_file_path': gt_path,
            'case_id': row.get('case_id'),
            'modality': row.get('modality'),
            'gt_type': row.get('gt_type'),
            'disc_or_slice_id': row.get('disc_or_slice_id'),
            'instance_number': row.get('instance_number'),
            'auto_sanity_status': sanity['auto_sanity_status'],
            'manual_accept': '',
            'manual_reject_reason': '',
            'notes': '',
        })
    except Exception as exc:
        sanity['error'] = repr(exc)
    sanity_rows.append(sanity)

curation_review_sheet_fast_df = pd.DataFrame(figure_rows)
candidate_sanity_fast_df = pd.DataFrame(sanity_rows)
review_path = CURATION_ROOT / 'E7_alkafri_axial_curation_review_sheet_FAST.csv'
sanity_path = CURATION_ROOT / 'E7_alkafri_axial_candidate_sanity_checks_FAST.csv'
curation_review_sheet_fast_df.to_csv(review_path, index=False)
candidate_sanity_fast_df.to_csv(sanity_path, index=False)
print('Figuras FAST generadas:', len(curation_review_sheet_fast_df))
print('Review sheet:', review_path)
print('Sanity checks:', sanity_path)
display(curation_review_sheet_fast_df.head())
display(candidate_sanity_fast_df.head())

auto_accept_count = 0
if len(candidate_sanity_fast_df) and 'auto_sanity_status' in candidate_sanity_fast_df.columns:
    auto_accept_count = int((candidate_sanity_fast_df['auto_sanity_status'] == 'ok').sum())

fast_report = {
    'mode': 'fast_patch',
    'cases_image': len(image_cases),
    'cases_gt': len(gt_cases),
    'common_cases': len(common_cases),
    'case_modality_overlap': int(len(case_modality_overlap_df)),
    'candidate_base_fast': int(len(candidate_base_df)),
    'strict_candidates_fast': int(len(strict_candidate_pairs_fast_df)),
    'visual_candidates_fast': int(len(curation_review_sheet_fast_df)),
    'auto_sanity_ok': auto_accept_count,
    'recommended_next_step': 'review_FAST_sheet_manually_then_create_curated_subset',
}
report_path = CURATION_ROOT / 'E7_alkafri_axial_fast_candidates_report.json'
report_path.write_text(json.dumps(fast_report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(fast_report, indent=2, ensure_ascii=False))


Candidatos para visualización FAST: 80


FAST visual sanity:   0%|          | 0/80 [00:00<?, ?it/s]

Figuras FAST generadas: 80
Review sheet: /content/drive/MyDrive/PFI_MVP/results/E7_alkafri_axial_curated_subset/E7_alkafri_axial_curation_review_sheet_FAST.csv
Sanity checks: /content/drive/MyDrive/PFI_MVP/results/E7_alkafri_axial_curated_subset/E7_alkafri_axial_candidate_sanity_checks_FAST.csv


,candidate_id,figure_path,image_file_path,gt_file_path,case_id,modality,gt_type,disc_or_slice_id,instance_number,auto_sanity_status,manual_accept,manual_reject_reason,notes
0,fast_candidate_001,/content/drive/MyDrive/PFI_MVP/figures/E7_alkafri_fast_curated_candidate_001.png,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0003/L-SPINE_LSS_20160411_134646_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,3,T2,intermediary,3,3,review,,,
1,fast_candidate_002,/content/drive/MyDrive/PFI_MVP/figures/E7_alkafri_fast_curated_candidate_002.png,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0030/L-SPINE_LSS_20160319_123721_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,30,T2,intermediary,4,2,review,,,
2,fast_candidate_003,/content/drive/MyDrive/PFI_MVP/figures/E7_alkafri_fast_curated_candidate_003.png,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0041/L-SPINE_LSS_20160112_154007_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,41,T2,intermediary,5,5,review,,,
3,fast_candidate_004,/content/drive/MyDrive/PFI_MVP/figures/E7_alkafri_fast_curated_candidate_004.png,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0026/L-SPINE_LSS_20151121_130729_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,26,T1,intermediary,4,1,review,,,
4,fast_candidate_005,/content/drive/MyDrive/PFI_MVP/figures/E7_alkafri_fast_curated_candidate_005.png,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0186/L-SPINE_CLINICAL_LIBRARIES_2...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,186,T1,intermediary,3,6,review,,,


,candidate_id,image_file_path,gt_file_path,case_id,modality,gt_type,disc_or_slice_id,instance_number,overlay_generated,auto_sanity_status,mask_empty,mask_full,foreground_ratio,component_count,image_shape_actual,mask_shape_actual,resize_needed
0,fast_candidate_001,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0003/L-SPINE_LSS_20160411_134646_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,3,T2,intermediary,3,3,True,review,False,True,0.968574,47,"(320, 320)","(320, 320)",False
1,fast_candidate_002,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0030/L-SPINE_LSS_20160319_123721_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,30,T2,intermediary,4,2,True,review,False,True,0.962627,92,"(320, 320)","(320, 320)",False
2,fast_candidate_003,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0041/L-SPINE_LSS_20160112_154007_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,41,T2,intermediary,5,5,True,review,False,True,0.973105,28,"(320, 320)","(320, 320)",False
3,fast_candidate_004,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0026/L-SPINE_LSS_20151121_130729_...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,26,T1,intermediary,4,1,True,review,False,True,0.958037,55,"(320, 320)","(320, 320)",False
4,fast_candidate_005,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/main_dataset__MRI_Data/01_MRI_Data/0186/L-SPINE_CLINICAL_LIBRARIES_2...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/_nested/ground_truth__Ground_Truth_Label/04_Intermediary_Ground_Truth_Data/T...,186,T1,intermediary,3,6,True,review,False,True,0.966914,74,"(320, 320)","(320, 320)",False


{
  "mode": "fast_patch",
  "cases_image": 185,
  "cases_gt": 515,
  "common_cases": 185,
  "case_modality_overlap": 369,
  "candidate_base_fast": 1200,
  "strict_candidates_fast": 1200,
  "visual_candidates_fast": 80,
  "auto_sanity_ok": 0,
  "recommended_next_step": "review_FAST_sheet_manually_then_create_curated_subset"
}
